In [1]:
import numpy as np
import math

In [2]:
from gensim.corpora import Dictionary
from src.detm import xDETM
from gensim.parsing.preprocessing import preprocess_string


In [3]:
from gensim.models import CoherenceModel
from gensim import matutils

"""def top_topics_as_word_lists(model, dictionary, topn=20):
        if not dictionary.id2token:
            dictionary.id2token = {v: k for k, v in dictionary.token2id.items()}
        if isinstance(model, xDETM):
            topics = model.get_topics(topn=topn)
            flattened_topics = [time_topic for topic in topics for time_topic in topic]
            return flattened_topics
        str_topics = []
        for topic in model.get_topics():
            bestn = matutils.argsort(topic, topn=topn, reverse=True)
            beststr = [dictionary.id2token[_id] for _id in bestn]
            str_topics.append(beststr)
        return str_topics

CoherenceModel.topics_as_word_lists = top_topics_as_word_lists"""


'def top_topics_as_word_lists(model, dictionary, topn=20):\n        if not dictionary.id2token:\n            dictionary.id2token = {v: k for k, v in dictionary.token2id.items()}\n        if isinstance(model, xDETM):\n            topics = model.get_topics(topn=topn)\n            flattened_topics = [time_topic for topic in topics for time_topic in topic]\n            return flattened_topics\n        str_topics = []\n        for topic in model.get_topics():\n            bestn = matutils.argsort(topic, topn=topn, reverse=True)\n            beststr = [dictionary.id2token[_id] for _id in bestn]\n            str_topics.append(beststr)\n        return str_topics\n\nCoherenceModel.topics_as_word_lists = top_topics_as_word_lists'

In [4]:
# create corpus generator
class DETMGenerator:
    def __init__(
            self,
            num_topics,
            embeddings,
            window_size,
            min_time,
            max_time,
            word_list,
            mixture_and_topic_deltas=(0.005, 0.005),
            mixture_variance=1,
    ):
        self.num_topics = num_topics
        self.embeddings = embeddings
        self.word_list = word_list
        embeddings_data = np.array([embeddings[w] for w in self.word_list])
        num_embeddings, embeddings_size = embeddings_data.shape
        self.num_embeddings = num_embeddings
        self.embeddings_size = embeddings_size
        self.embeddings_data = embeddings_data
        self.window_size = window_size
        self.num_windows = math.ceil((max_time - min_time) / window_size)
        self.min_time = min_time
        self.max_time = max_time
        self.mixture_delta = mixture_and_topic_deltas[0]
        self.topic_delta = mixture_and_topic_deltas[1]
        self.mixture_variance = mixture_variance
        self.initialize()
    
    def random_gaussian_step(self, vector, delta):
        return vector + np.random.normal(0, delta, vector.shape)

    def random_gaussian_walk(self, vector, delta, steps=1):
        vectors= [vector]
        for i in range(1, steps):
            vectors.append(self.random_gaussian_step(vectors[-1], delta))
        return vectors

    def initialize(self):
        # to generate: 
        # topic vectors = self.num_topics x self.window_size x self.embeddings_size
        # mixture vector = self.num_topics x self.window_size
        # generate topic vectors like this:
        # create for window zero choose a vector from the N(0, 1) distribution, then walk a gaussian random walk with delta self.topic_delta
        # generate mixture vector like this:
        # create for window zero choose a vector from the N(0, 1) distribution, then walk a gaussian random walk with delta self.mixture_delta
        initial_topic_vectors = np.random.normal(0, 1, (self.num_topics, self.embeddings_size))
        self.topic_vectors = self.random_gaussian_walk(initial_topic_vectors, self.topic_delta, self.num_windows)
        self.topic_vectors = np.stack(self.topic_vectors, axis=0)

        flattened_topics = self.topic_vectors.reshape(self.num_windows * self.num_topics, self.embeddings_size)
        word_probabilities = np.dot(flattened_topics, self.embeddings_data.T)
        word_probabilities = np.exp(word_probabilities) / np.sum(np.exp(word_probabilities), axis=1, keepdims=True)
        word_probabilities = word_probabilities.reshape(self.num_windows, self.num_topics, self.num_embeddings)
        self.word_probabilities = word_probabilities

        initial_mixture_vector = np.random.normal(0, 1, (self.num_topics,))
        self.mixture_vector = self.random_gaussian_walk(initial_mixture_vector, self.mixture_delta, self.num_windows)
        self.mixture_vector = np.stack(self.mixture_vector, axis=0)

    def generate_document(self, subdoc_size, time_window):
        document = []
        mixture_vector = self.mixture_vector[time_window]
        temp = np.random.normal(mixture_vector, self.mixture_variance)
        document_mixture_vector = np.exp(temp) / np.sum(np.exp(temp))
        topic_counts = [0] * self.num_topics
        for word in range(subdoc_size):
            topic = np.random.choice(self.num_topics, p=document_mixture_vector)
            word_probabilities_temp = self.word_probabilities[time_window,topic, :]
            word = np.random.choice(self.word_list, p=word_probabilities_temp)
            document.append(word)
            topic_counts[topic] += 1
        return (document, topic_counts, mixture_vector)
            
        


    def generate(self, subdoc_size):
        # generate subdoc_size bag of word documents
        # assume the time window is sampled from a uniform distribution
        # assume the topics per document is sampled from the time slices mixture vector + some noise with mixture_variance and then softmaxed

        # TODO: ask Tom why the thetas have such a high varience? 

        # precompute word probabilities

        while True:
            time_window = int(np.random.uniform(0, self.num_windows))

            document, topic_counts, mixture = self.generate_document(subdoc_size, time_window)

            yield (document, time_window, topic_counts, mixture)

In [5]:
# import gensim api

import gensim.downloader as api
import random

In [6]:
w2v = api.load("word2vec-google-news-300")

In [7]:
word_list = random.sample(list(w2v.index_to_key), 10000)

In [8]:
downloaded_corpus = api.load("20-newsgroups")

In [9]:
corpus = api.load('20-newsgroups')
text = [
    preprocess_string(text['data'])
    for text in corpus
]

In [10]:
dictioary = Dictionary(text)

In [11]:
dictionary_word_list = dictioary.token2id.keys()

In [12]:
words = set(dictionary_word_list).intersection(set(w2v.index_to_key))

In [13]:
words = list(words)

In [14]:
xDETM.get_topic_words

<function src.detm.abstract_detm.AbstractDETM.get_topic_words(self, topk=10, topic_embeddings=None)>

In [15]:
model = xDETM(
    num_topics=10,
    embeddings=w2v,
    window_size=10,
    min_time=0,
    max_time=50,
    word_list=words,
)

In [16]:
from src.detm.evaluations import *

In [17]:
model.topic_distributions()

xDETM doesn't create per-document topic representations!


tensor([[[4.9937e-08, 1.2148e-09, 2.4109e-10,  ..., 8.7690e-04,
          2.3249e-08, 1.7584e-09],
         [1.0729e-12, 1.4895e-10, 8.7546e-10,  ..., 3.8822e-11,
          7.2729e-14, 1.2418e-13],
         [1.2445e-09, 1.3553e-11, 3.6077e-11,  ..., 7.1243e-07,
          6.2604e-07, 1.4358e-08],
         ...,
         [1.9496e-10, 8.9756e-08, 3.1528e-10,  ..., 8.9488e-12,
          3.1536e-09, 1.4465e-06],
         [1.0769e-12, 1.0637e-14, 5.9089e-15,  ..., 7.4810e-13,
          9.4651e-12, 1.3130e-17],
         [8.1555e-11, 1.2924e-13, 2.0749e-15,  ..., 1.2961e-11,
          3.0513e-08, 6.0053e-08]],

        [[2.0377e-11, 8.9398e-12, 7.2605e-13,  ..., 3.8939e-09,
          6.4757e-11, 3.8215e-11],
         [9.9864e-10, 7.1069e-09, 1.4039e-10,  ..., 8.2966e-11,
          1.0067e-10, 1.1610e-11],
         [5.0078e-11, 5.0712e-11, 4.3041e-12,  ..., 1.6140e-08,
          5.2291e-11, 3.8496e-11],
         ...,
         [2.5011e-10, 2.9888e-09, 3.7599e-10,  ..., 1.0707e-11,
          6.366

In [18]:

corpus = api.load('20-newsgroups')
text = [
    preprocess_string(text['data'])
    for text in corpus
    if text['topic'] in ('soc.religion.christian', 'talk.politics.guns')
]
dictionary = Dictionary(text)

In [19]:
dictionary

In [20]:
text = api.load('text8')
dictionary2 = Dictionary(text)

In [21]:
dictionary2

In [22]:
"accept" in dictionary

False

In [23]:
dictionary.token2id

{'accept': 0,
 'action': 1,
 'adulter': 2,
 'adulteri': 3,
 'adventur': 4,
 'affect': 5,
 'andi': 6,
 'andrew': 7,
 'annia': 8,
 'bar': 9,
 'begin': 10,
 'believ': 11,
 'bibl': 12,
 'biblic': 13,
 'bless': 14,
 'bosnia': 15,
 'byler': 16,
 'calvinist': 17,
 'cannanit': 18,
 'carnegi': 19,
 'case': 20,
 'certainli': 21,
 'chastis': 22,
 'child': 23,
 'choos': 24,
 'chose': 25,
 'chosen': 26,
 'christian': 27,
 'church': 28,
 'civil': 29,
 'cmu': 30,
 'correct': 31,
 'cours': 32,
 'creator': 33,
 'david': 34,
 'dbn': 35,
 'death': 36,
 'deni': 37,
 'deserv': 38,
 'differ': 39,
 'disagre': 40,
 'dispos': 41,
 'doctrin': 42,
 'edu': 43,
 'effort': 44,
 'elect': 45,
 'end': 46,
 'engin': 47,
 'everlast': 48,
 'everybodi': 49,
 'excus': 50,
 'final': 51,
 'foreign': 52,
 'foreknew': 53,
 'forgiven': 54,
 'frankli': 55,
 'free': 56,
 'freeli': 57,
 'freshman': 58,
 'fundament': 59,
 'furthermor': 60,
 'gener': 61,
 'genocid': 62,
 'get': 63,
 'gift': 64,
 'give': 65,
 'go': 66,
 'god': 67,
 '

In [24]:

corpus = api.load('20-newsgroups')
text = [
    preprocess_string(text['data'])
    for text in corpus
]

dictionary = Dictionary(text)


In [25]:
topics = model.get_topic_words(10)

xDETM doesn't create per-document topic representations!


In [26]:
len(dictionary)

85091

In [27]:
"underinvestment" in dictionary.token2id

False

In [28]:
len(dictionary)

85091

In [29]:
#topics = [[['disrepair', 'rots', 'underinvestment', 'histoplasmosis', 'typos', 'parses', 'unsaved', 'receivership', 'abbreviations', 'nasdaq'], ['licked', 'starves', 'brimstone', 'lapdog', 'oligarchy', 'pinstriped', 'rickets', 'autocrat', 'yob', 'ayatollah'], ['deflator', 'kl', 'incandescents', 'stanozolol', 'billionths', 'pardoning', 'disclaims', 'wedlock', 'tritium', 'kilotons'], ['flanker', 'ruckman', 'salesforce', 'selectors', 'spirituals', 'neutrals', 'touchbacks', 'playmaker', 'wideout', 'mead'], ['biopharmaceutical', 'therapeutics', 'franks', 'cadmium', 'monkeypox', 'anaphylaxis', 'nutritionals', 'salmonellosis', 'cume', 'excreted'], ['wheelbase', 'legroom', 'snowmobilers', 'mpg', 'supermini', 'shearer', 'hatchback', 'yachtsmen', 'millimeters', 'minigames'], ['chartists', 'flammability', 'demographers', 'cortisone', 'glaciologist', 'quarks', 'carcinogen', 'acetaminophen', 'desegregated', 'seismologist'], ['transliterated', 'vocalizations', 'dialect', 'allele', 'fraternizing', 'honorific', 'bibliography', 'adverb', 'naproxen', 'gunship'], ['ultramafic', 'capt', 'breccia', 'spud', 'pallbearers', 'howitzers', 'paydirt', 'resigns', 'vanadium', 'ebitda'], ['gainers', 'liaising', 'tribes', 'skirmished', 'ransoms', 'interoperable', 'username', 'crossbones', 'honored', 'felicitation']], [['bushels', 'minyan', 'withers', 'pastoralists', 'transpiration', 'birr', 'herbalists', 'pilgrimages', 'shillings', 'heuristics'], ['apnea', 'matchday', 'metastasis', 'lymphoma', 'idents', 'microenvironment', 'ischemia', 'thorax', 'dancefloor', 'qi'], ['smooths', 'pips', 'neckline', 'undocking', 'mintage', 'fractionation', 'cheekbone', 'kroon', 'decriminalized', 'cyclosporine'], ['neutropenia', 'lifecycle', 'janjaweed', 'residuals', 'timeouts', 'coltan', 'believability', 'counteraction', 'orgasms', 'battlespace'], ['xvii', 'biodegrade', 'oceanographer', 'emulsifier', 'lubricity', 'xiv', 'seabirds', 'macrophages', 'chlorofluorocarbons', 'playback'], ['billionths', 'solicitation', 'indecency', 'tabular', 'uncollectible', 'zakat', 'clerics', 'penitence', 'macula', 'clothesline'], ['kilotons', 'sr', 'effigies', 'hefted', 'densities', 'girdles', 'couplings', 'effigy', 'kidnappings', 'sweeten'], ['batik', 'silversmith', 'blacksmiths', 'crucifixes', 'silversmiths', 'unframed', 'repaint', 'carver', 'mantel', 'cupola'], ['disclaims', 'recency', 'tael', 'speechwriting', 'exercisable', 'cume', 'depositary', 'rabies', 'flied', 'layover'], ['transmittal', 'saucepan', 'cayenne', 'televoting', 'commas', 'undervote', 'suk', 'breasted', 'stowage', 'condominium']], [['nasdaq', 'vulgarities', 'overeat', 'firebombs', 'bugle', 'glycogen', 'snowiest', 'headline', 'filename', 'birthrates'], ['captivity', 'captors', 'miser', 'barometers', 'counterrevolutionary', 'unrevised', 'restocked', 'twiddling', 'await', 'doted'], ['stagehands', 'acrylamide', 'papercuts', 'chambermaid', 'twisters', 'cartoonists', 'necropsy', 'bluefish', 'kingfish', 'shrimpers'], ['suplex', 'unsubscribe', 'systolic', 'semiretired', 'percentiles', 'interstitials', 'cheesemakers', 'bigamy', 'diacetyl', 'dieter'], ['mineralization', 'mineralized', 'stratigraphic', 'mafic', 'sediments', 'meteorites', 'calmest', 'geochemical', 'anticline', 'sedimentary'], ['transistors', 'mastitis', 'downwards', 'nanometers', 'kyat', 'fluctuate', 'triceps', 'chartists', 'overbought', 'dma'], ['groundstrokes', 'gd', 'pistachios', 'deuce', 'gf', 'volleyed', 'tiebreak', 'feuded', 'overthrew', 'deported'], ['ossuary', 'chronograph', 'quilter', 'reenactors', 'punishable', 'ossuaries', 'engraved', 'locket', 'disclaims', 'blacksmithing'], ['midshipmen', 'rx', 'creeds', 'amortized', 'hematologic', 'engines', 'shp', 'erections', 'narrowband', 'mayday'], ['cadmium', 'frankincense', 'canto', 'kingdom', 'piracy', 'rubies', 'boxoffice', 'sultanate', 'genies', 'supertanker']], [['perchlorate', 'tritium', 'keno', 'nakba', 'misconducts', 'oneworld', 'redeployments', 'renames', 'infringes', 'reauthorized'], ['earshot', 'sensitively', 'outcroppings', 'pencils', 'ruffling', 'goldmine', 'slickly', 'immaculately', 'selflessly', 'venturesome'], ['predictor', 'scapegoat', 'qtr', 'deckchairs', 'camaraderie', 'boon', 'televises', 'cardiomyopathy', 'analogy', 'capt'], ['conscripting', 'circumcise', 'memoir', 'paparazzo', 'sirloin', 'prospector', 'swindler', 'diamonds', 'backdating', 'vintner'], ['sr', 'homered', 'refiner', 'aet', 'twirled', 'dec', 'rw', 'naphtha', 'myrrh', 'playout'], ['decontaminated', 'shrimping', 'mononucleosis', 'cysts', 'spacewalkers', 'shrimpers', 'pallbearers', 'pollutants', 'claustrophobia', 'canneries'], ['merchantability', 'passcode', 'disclaims', 'linebacker', 'multispectral', 'cornerback', 'winger', 'ballcarrier', 'megawatt', 'turnovers'], ['windfarm', 'polygamist', 'jeera', 'scrummage', 'bmibaby', 'iwi', 'supermini', 'paleontologist', 'deflator', 'teledensity'], ['commas', 'slits', 'retirements', 'reshuffles', 'upsets', 'departures', 'epiphanies', 'mussed', 'scribblings', 'subtractions'], ['magnate', 'cume', 'synaptic', 'erectile', 'erections', 'agonist', 'placebo', 'mediates', 'apoptotic', 'synapses']], [['spacewalks', 'exoplanets', 'condors', 'habitability', 'psilocybin', 'exoplanet', 'supernovae', 'chinook', 'petabyte', 'multicenter'], ['smooths', 'endorphins', 'cinematographer', 'neurotransmitters', 'capillaries', 'virtuosos', 'impracticable', 'lothario', 'neurobiologist', 'soulfulness'], ['downforce', 'endorphins', 'subindex', 'humidity', 'benchmark', 'superstardom', 'eur', 'stardom', 'peso', 'mintage'], ['astronomers', 'spacewalker', 'redshift', 'stent', 'orbiter', 'astrobiology', 'astronomer', 'fainter', 'orbiters', 'magnetometer'], ['layup', 'legdrop', 'downforce', 'cruiserweight', 'geophysicist', 'bunt', 'integer', 'dunk', 'spacewalker', 'composited'], ['aqueduct', 'repopulation', 'electrification', 'wideband', 'umbilical', 'rectifier', 'ecozone', 'infocomm', 'spillways', 'dispersant'], ['warlord', 'booms', 'wah', 'loots', 'swindler', 'hijacker', 'kee', 'abducts', 'evildoer', 'grenade'], ['gapped', 'easement', 'embankment', 'loped', 'camber', 'faller', 'impracticable', 'limped', 'sprinkle', 'reusability'], ['smooths', 'flied', 'hitless', 'lf', 'ss', 'rf', 'consoler', 'deflator', 'scoreless', 'bunted'], ['copywriter', 'plagiarize', 'republication', 'trowel', 'napkin', 'demoting', 'holeshot', 'colorist', 'predeceased', 'publisher']]]


In [30]:
window = 1
coherence_measure = "c_v"
topn = 10
topics = model.get_topic_words(topn)
#topics = [topic for window_topic in topics for topic in window_topic]
print(topics)
num_windows = len(topics)
coherences = {}

for topic in topics[0]:
    count = 0
    for word in topic:
        if word in dictionary.token2id:
            count += 1
            #print(word)
        else :
            print(f"word {word} not in dictionary")
    print(count)

coherence_model = CoherenceModel(topics=topics[window], texts=text, coherence=coherence_measure, topn=topn, dictionary=dictionary)
coherences[window] = coherence_model.get_coherence()

xDETM doesn't create per-document topic representations!


[[['libido', 'strewn', 'cultism', 'tik', 'reflux', 'doin', 'shiva', 'tryin', 'hypochondriac', 'alli'], ['defrag', 'quorum', 'counterclaim', 'databook', 'orbit', 'thruster', 'astronaut', 'subpoena', 'jumper', 'diva'], ['databook', 'forehand', 'survivalist', 'putt', 'thrombocytopenia', 'samurai', 'kimura', 'pickax', 'sharpen', 'sinew'], ['reverb', 'dogleg', 'carbon', 'login', 'sulfur', 'republish', 'emit', 'shutter', 'downwind', 'nudist'], ['twister', 'easement', 'ecologist', 'unseal', 'antitrust', 'serotonin', 'boreal', 'unbroken', 'drought', 'streamflow'], ['maj', 'dec', 'easement', 'exon', 'photon', 'fullscreen', 'filer', 'trier', 'pulsar', 'mpeg'], ['plea', 'pleas', 'prohibit', 'effluent', 'suburbia', 'forbid', 'denser', 'prettier', 'fewest', 'underfund'], ['loge', 'skywalk', 'dwelt', 'flab', 'kurta', 'cropper', 'piggyback', 'sleeveless', 'cess', 'viral'], ['kashrut', 'jodi', 'hajj', 'groundout', 'orca', 'tigress', 'rhp', 'album', 'kosher', 'colt'], ['aorta', 'shekel', 'kilowatt', 'm

In [31]:
text=None

In [32]:
if text is None:
    corpus = api.load('20-newsgroups')
    text = [
        preprocess_string(text['data'])
        for text in corpus
    ]

dictionary = Dictionary(text)

In [33]:
topics = model.get_topic_words(topn)
#topics = [topic for window in topics for topic in window]
print(topics)
num_windows = len(topics)
coherences = {}
for window in range(num_windows):
    try:
        coherence_model = CoherenceModel(topics=topics[window], texts=text, coherence=coherence_measure, topn=topn, dictionary=dictionary)
        coherences[window] = coherence_model.get_coherence()
    except:
        raise Exception(f'Error in coherence calculation in window {window}')

xDETM doesn't create per-document topic representations!


[[['libido', 'strewn', 'cultism', 'tik', 'reflux', 'doin', 'shiva', 'tryin', 'hypochondriac', 'alli'], ['defrag', 'quorum', 'counterclaim', 'databook', 'orbit', 'thruster', 'astronaut', 'subpoena', 'jumper', 'diva'], ['databook', 'forehand', 'survivalist', 'putt', 'thrombocytopenia', 'samurai', 'kimura', 'pickax', 'sharpen', 'sinew'], ['reverb', 'dogleg', 'carbon', 'login', 'sulfur', 'republish', 'emit', 'shutter', 'downwind', 'nudist'], ['twister', 'easement', 'ecologist', 'unseal', 'antitrust', 'serotonin', 'boreal', 'unbroken', 'drought', 'streamflow'], ['maj', 'dec', 'easement', 'exon', 'photon', 'fullscreen', 'filer', 'trier', 'pulsar', 'mpeg'], ['plea', 'pleas', 'prohibit', 'effluent', 'suburbia', 'forbid', 'denser', 'prettier', 'fewest', 'underfund'], ['loge', 'skywalk', 'dwelt', 'flab', 'kurta', 'cropper', 'piggyback', 'sleeveless', 'cess', 'viral'], ['kashrut', 'jodi', 'hajj', 'groundout', 'orca', 'tigress', 'rhp', 'album', 'kosher', 'colt'], ['aorta', 'shekel', 'kilowatt', 'm

In [34]:
evaluate_coherence(model, coherence_measure="c_v")

xDETM doesn't create per-document topic representations!


[[['libido', 'strewn', 'cultism', 'tik', 'reflux', 'doin', 'shiva', 'tryin', 'hypochondriac', 'alli'], ['defrag', 'quorum', 'counterclaim', 'databook', 'orbit', 'thruster', 'astronaut', 'subpoena', 'jumper', 'diva'], ['databook', 'forehand', 'survivalist', 'putt', 'thrombocytopenia', 'samurai', 'kimura', 'pickax', 'sharpen', 'sinew'], ['reverb', 'dogleg', 'carbon', 'login', 'sulfur', 'republish', 'emit', 'shutter', 'downwind', 'nudist'], ['twister', 'easement', 'ecologist', 'unseal', 'antitrust', 'serotonin', 'boreal', 'unbroken', 'drought', 'streamflow'], ['maj', 'dec', 'easement', 'exon', 'photon', 'fullscreen', 'filer', 'trier', 'pulsar', 'mpeg'], ['plea', 'pleas', 'prohibit', 'effluent', 'suburbia', 'forbid', 'denser', 'prettier', 'fewest', 'underfund'], ['loge', 'skywalk', 'dwelt', 'flab', 'kurta', 'cropper', 'piggyback', 'sleeveless', 'cess', 'viral'], ['kashrut', 'jodi', 'hajj', 'groundout', 'orca', 'tigress', 'rhp', 'album', 'kosher', 'colt'], ['aorta', 'shekel', 'kilowatt', 'm

(0.4618188883777061,
 {0: 0.46700765343608736,
  1: 0.4744652438067199,
  2: 0.45497576659673794,
  3: 0.45765507771627406,
  4: 0.4549907003327113})

In [35]:
evaluate_coherence(model, coherence_measure="c_uci")

xDETM doesn't create per-document topic representations!


[[['libido', 'strewn', 'cultism', 'tik', 'reflux', 'doin', 'shiva', 'tryin', 'hypochondriac', 'alli'], ['defrag', 'quorum', 'counterclaim', 'databook', 'orbit', 'thruster', 'astronaut', 'subpoena', 'jumper', 'diva'], ['databook', 'forehand', 'survivalist', 'putt', 'thrombocytopenia', 'samurai', 'kimura', 'pickax', 'sharpen', 'sinew'], ['reverb', 'dogleg', 'carbon', 'login', 'sulfur', 'republish', 'emit', 'shutter', 'downwind', 'nudist'], ['twister', 'easement', 'ecologist', 'unseal', 'antitrust', 'serotonin', 'boreal', 'unbroken', 'drought', 'streamflow'], ['maj', 'dec', 'easement', 'exon', 'photon', 'fullscreen', 'filer', 'trier', 'pulsar', 'mpeg'], ['plea', 'pleas', 'prohibit', 'effluent', 'suburbia', 'forbid', 'denser', 'prettier', 'fewest', 'underfund'], ['loge', 'skywalk', 'dwelt', 'flab', 'kurta', 'cropper', 'piggyback', 'sleeveless', 'cess', 'viral'], ['kashrut', 'jodi', 'hajj', 'groundout', 'orca', 'tigress', 'rhp', 'album', 'kosher', 'colt'], ['aorta', 'shekel', 'kilowatt', 'm

(-6.024247888077807,
 {0: -6.18491201426558,
  1: -6.10612002582136,
  2: -5.7613173663675505,
  3: -6.078351902930869,
  4: -5.990538131003673})

In [36]:
evaluate_coherence(model, coherence_measure="c_npmi")

xDETM doesn't create per-document topic representations!


[[['libido', 'strewn', 'cultism', 'tik', 'reflux', 'doin', 'shiva', 'tryin', 'hypochondriac', 'alli'], ['defrag', 'quorum', 'counterclaim', 'databook', 'orbit', 'thruster', 'astronaut', 'subpoena', 'jumper', 'diva'], ['databook', 'forehand', 'survivalist', 'putt', 'thrombocytopenia', 'samurai', 'kimura', 'pickax', 'sharpen', 'sinew'], ['reverb', 'dogleg', 'carbon', 'login', 'sulfur', 'republish', 'emit', 'shutter', 'downwind', 'nudist'], ['twister', 'easement', 'ecologist', 'unseal', 'antitrust', 'serotonin', 'boreal', 'unbroken', 'drought', 'streamflow'], ['maj', 'dec', 'easement', 'exon', 'photon', 'fullscreen', 'filer', 'trier', 'pulsar', 'mpeg'], ['plea', 'pleas', 'prohibit', 'effluent', 'suburbia', 'forbid', 'denser', 'prettier', 'fewest', 'underfund'], ['loge', 'skywalk', 'dwelt', 'flab', 'kurta', 'cropper', 'piggyback', 'sleeveless', 'cess', 'viral'], ['kashrut', 'jodi', 'hajj', 'groundout', 'orca', 'tigress', 'rhp', 'album', 'kosher', 'colt'], ['aorta', 'shekel', 'kilowatt', 'm

(-0.21638400566456212,
 {0: -0.2206200895159518,
  1: -0.2209161585696438,
  2: -0.20682982742755085,
  3: -0.21916207526546563,
  4: -0.21439187754419864})

In [37]:
evaluate_coherence(model, coherence_measure="u_mass")

xDETM doesn't create per-document topic representations!


[[['libido', 'strewn', 'cultism', 'tik', 'reflux', 'doin', 'shiva', 'tryin', 'hypochondriac', 'alli'], ['defrag', 'quorum', 'counterclaim', 'databook', 'orbit', 'thruster', 'astronaut', 'subpoena', 'jumper', 'diva'], ['databook', 'forehand', 'survivalist', 'putt', 'thrombocytopenia', 'samurai', 'kimura', 'pickax', 'sharpen', 'sinew'], ['reverb', 'dogleg', 'carbon', 'login', 'sulfur', 'republish', 'emit', 'shutter', 'downwind', 'nudist'], ['twister', 'easement', 'ecologist', 'unseal', 'antitrust', 'serotonin', 'boreal', 'unbroken', 'drought', 'streamflow'], ['maj', 'dec', 'easement', 'exon', 'photon', 'fullscreen', 'filer', 'trier', 'pulsar', 'mpeg'], ['plea', 'pleas', 'prohibit', 'effluent', 'suburbia', 'forbid', 'denser', 'prettier', 'fewest', 'underfund'], ['loge', 'skywalk', 'dwelt', 'flab', 'kurta', 'cropper', 'piggyback', 'sleeveless', 'cess', 'viral'], ['kashrut', 'jodi', 'hajj', 'groundout', 'orca', 'tigress', 'rhp', 'album', 'kosher', 'colt'], ['aorta', 'shekel', 'kilowatt', 'm

(-18.5850459301618,
 {0: -18.671390544462735,
  1: -18.2806730557298,
  2: -18.492531880649715,
  3: -18.845549193983977,
  4: -18.63508497598277})

In [38]:
evaluate_coherence(model, coherence_measure="c_w2v")

xDETM doesn't create per-document topic representations!


[[['libido', 'strewn', 'cultism', 'tik', 'reflux', 'doin', 'shiva', 'tryin', 'hypochondriac', 'alli'], ['defrag', 'quorum', 'counterclaim', 'databook', 'orbit', 'thruster', 'astronaut', 'subpoena', 'jumper', 'diva'], ['databook', 'forehand', 'survivalist', 'putt', 'thrombocytopenia', 'samurai', 'kimura', 'pickax', 'sharpen', 'sinew'], ['reverb', 'dogleg', 'carbon', 'login', 'sulfur', 'republish', 'emit', 'shutter', 'downwind', 'nudist'], ['twister', 'easement', 'ecologist', 'unseal', 'antitrust', 'serotonin', 'boreal', 'unbroken', 'drought', 'streamflow'], ['maj', 'dec', 'easement', 'exon', 'photon', 'fullscreen', 'filer', 'trier', 'pulsar', 'mpeg'], ['plea', 'pleas', 'prohibit', 'effluent', 'suburbia', 'forbid', 'denser', 'prettier', 'fewest', 'underfund'], ['loge', 'skywalk', 'dwelt', 'flab', 'kurta', 'cropper', 'piggyback', 'sleeveless', 'cess', 'viral'], ['kashrut', 'jodi', 'hajj', 'groundout', 'orca', 'tigress', 'rhp', 'album', 'kosher', 'colt'], ['aorta', 'shekel', 'kilowatt', 'm

(0.827575,
 {0: 0.8202941, 1: 0.83702505, 2: 0.8252427, 3: 0.82538545, 4: 0.8299279})

In [39]:
for diversity_measure in ['proportion_unique_words', 'irbo', 'word_embedding_irbo', 'pairwise_jaccard_diversity', 'pairwise_word_embedding_distance', 'centroid_distance']:
    print(f"diversity_measure: {diversity_measure}")
    print(evaluate_topic_diversity(model, diversity_measure, embedding=w2v))

xDETM doesn't create per-document topic representations!
xDETM doesn't create per-document topic representations!
xDETM doesn't create per-document topic representations!


diversity_measure: proportion_unique_words
(0.9640000000000001, {0: 0.98, 1: 0.96, 2: 0.98, 3: 0.94, 4: 0.96})
diversity_measure: irbo
(0.9919252898965396, {0: 0.9950318114028571, 1: 0.9881645604620635, 2: 0.99803898271, 3: 0.9824194176877777, 4: 0.99597167722})
diversity_measure: word_embedding_irbo


xDETM doesn't create per-document topic representations!
xDETM doesn't create per-document topic representations!


(0.859920251353202, {0: 0.8696392295370516, 1: 0.8482247145840888, 2: 0.8604517688466299, 3: 0.8484060459123688, 4: 0.872879497885871})
diversity_measure: pairwise_jaccard_diversity
(0.9955295646523716, {0: 0.9976608187134502, 1: 0.9951916829109811, 2: 0.9976608187134502, 3: 0.9918128654970759, 4: 0.9953216374269005})
diversity_measure: pairwise_word_embedding_distance


xDETM doesn't create per-document topic representations!


(0.9228598845849559, {0: 0.9275591148480848, 1: 0.9174769053072283, 2: 0.9219635750394859, 3: 0.918870975693355, 4: 0.9284288520366253})
diversity_measure: centroid_distance
(0.6422217702904931, {0: 0.5735585497866307, 1: 0.7524796457423053, 2: 0.6186104467114142, 3: 0.597870027454051, 4: 0.668590181758064})


In [40]:
evaluate_topic_diversity(model,"proportion_unique_words")

xDETM doesn't create per-document topic representations!


(0.9640000000000001, {0: 0.98, 1: 0.96, 2: 0.98, 3: 0.94, 4: 0.96})

In [41]:
evaluate_topic_diversity(model,"irbo")

xDETM doesn't create per-document topic representations!


(0.9919252898965396,
 {0: 0.9950318114028571,
  1: 0.9881645604620635,
  2: 0.99803898271,
  3: 0.9824194176877777,
  4: 0.99597167722})

In [42]:
evaluate_topic_diversity(model,"word_embedding_irbo", embedding=w2v)

xDETM doesn't create per-document topic representations!


(0.859920251353202,
 {0: 0.8696392295370516,
  1: 0.8482247145840888,
  2: 0.8604517688466299,
  3: 0.8484060459123688,
  4: 0.872879497885871})

In [43]:
evaluate_topic_diversity(model,"pairwise_jaccard_diversity")

xDETM doesn't create per-document topic representations!


(0.9955295646523716,
 {0: 0.9976608187134502,
  1: 0.9951916829109811,
  2: 0.9976608187134502,
  3: 0.9918128654970759,
  4: 0.9953216374269005})

In [44]:
print(model.get_topic_words())

xDETM doesn't create per-document topic representations!


[[['libido', 'strewn', 'cultism', 'tik', 'reflux', 'doin', 'shiva', 'tryin', 'hypochondriac', 'alli'], ['defrag', 'quorum', 'counterclaim', 'databook', 'orbit', 'thruster', 'astronaut', 'subpoena', 'jumper', 'diva'], ['databook', 'forehand', 'survivalist', 'putt', 'thrombocytopenia', 'samurai', 'kimura', 'pickax', 'sharpen', 'sinew'], ['reverb', 'dogleg', 'carbon', 'login', 'sulfur', 'republish', 'emit', 'shutter', 'downwind', 'nudist'], ['twister', 'easement', 'ecologist', 'unseal', 'antitrust', 'serotonin', 'boreal', 'unbroken', 'drought', 'streamflow'], ['maj', 'dec', 'easement', 'exon', 'photon', 'fullscreen', 'filer', 'trier', 'pulsar', 'mpeg'], ['plea', 'pleas', 'prohibit', 'effluent', 'suburbia', 'forbid', 'denser', 'prettier', 'fewest', 'underfund'], ['loge', 'skywalk', 'dwelt', 'flab', 'kurta', 'cropper', 'piggyback', 'sleeveless', 'cess', 'viral'], ['kashrut', 'jodi', 'hajj', 'groundout', 'orca', 'tigress', 'rhp', 'album', 'kosher', 'colt'], ['aorta', 'shekel', 'kilowatt', 'm

In [45]:
model.dictionary = dictioary

In [46]:
downloaded_corpus.__dict__

{'fn': '/home/efittsc1/gensim-data/20-newsgroups/20-newsgroups.gz'}

In [ ]:
generator = DETMGenerator(
    num_topics=10,
    embeddings=w2v,
    window_size=10,
    min_time=0,
    max_time=50,
    word_list=words,
)

In [ ]:
generator.num_windows

5

In [ ]:
generator.topic_vectors.shape

(5, 10, 300)

In [ ]:
generator.generate_document(100, 0)

(['orderliness',
  'gs',
  'volcanos',
  'impracticable',
  'knighthoods',
  'impracticable',
  'curable',
  'impracticable',
  'attache',
  'impracticable',
  'laughable',
  'offeree',
  'bombers',
  'submersibles',
  'volcanos',
  'ungrounded',
  'volcanos',
  'impracticable',
  'alcoholics',
  'libelous',
  'cortege',
  'pings',
  'volcanos',
  'libelous',
  'ricin',
  'volcanos',
  'handsprings',
  'addictions',
  'datacom',
  'binomial',
  'hydrazine',
  'libelous',
  'inappropriately',
  'shrimping',
  'propellant',
  'volcanos',
  'outraised',
  'impracticable',
  'libelous',
  'gd',
  'impracticable',
  'skillet',
  'shrimping',
  'intravenously',
  'uppercuts',
  'agoraphobia',
  'autobiography',
  'curable',
  'volcanos',
  'volcanos',
  'volcanos',
  'impracticable',
  'stove',
  'pings',
  'impracticable',
  'brucellosis',
  'pings',
  'pings',
  'ribavirin',
  'lexapro',
  'digoxin',
  'volcanos',
  'gd',
  'libelous',
  'conclave',
  'exasperating',
  'sr',
  'pings',
  '